## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Pizza taste test: crust, cheese and price tag

**Factorial Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A pizza chain ran a blind taste test with three crusts, regular or extra cheese, and a menu card showing either \$8 or \$14. The pizza was the same whatever the price on the card. Each combination was rated by three tasters.

**The question.** Which of crust, cheese and price tag change how much people like the pizza?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/factorial_anova.csv` (36 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Crust,Cheese,Price tag,Rating
Thin,Regular,$8,5.0
Thin,Regular,$8,7.0
Thin,Regular,$8,6.0
Thin,Regular,$14,8.0
Thin,Regular,$14,6.0
Thin,Regular,$14,7.0
Thin,Extra,$8,8.0
Thin,Extra,$8,6.0
Thin,Extra,$8,6.0
Thin,Extra,$14,9.0
Thin,Extra,$14,8.0
Thin,Extra,$14,8.0
Thick,Regular,$8,6.0
Thick,Regular,$8,5.0
Thick,Regular,$8,6.0
Thick,Regular,$14,6.0
Thick,Regular,$14,8.0
Thick,Regular,$14,6.0
Thick,Extra,$8,6.0
Thick,Extra,$8,6.0
Thick,Extra,$8,7.0
Thick,Extra,$14,7.0
Thick,Extra,$14,7.0
Thick,Extra,$14,8.0
Stuffed,Regular,$8,5.0
Stuffed,Regular,$8,6.0
Stuffed,Regular,$8,5.0
Stuffed,Regular,$14,7.0
Stuffed,Regular,$14,7.0
Stuffed,Regular,$14,8.0
Stuffed,Extra,$8,7.0
Stuffed,Extra,$8,6.0
Stuffed,Extra,$8,8.0
Stuffed,Extra,$14,8.0
Stuffed,Extra,$14,8.0
Stuffed,Extra,$14,9.0
""")

## Helper: eta_squared_interval

A step that no library does, written out. In the code download it is the file `eta_squared_interval.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response and one column per factor. A row with a blank
# in any of these columns is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Rating"
factors = ["Crust", "Cheese", "Price tag"]
data = data[[response, *factors]].dropna()
# Each factor's levels in the order they first appear in the table, as the app lists them.
for name in factors:
    data[name] = pd.Categorical(data[name], categories=pd.unique(data[name]))

## The model

In [ ]:
# Every main effect and every interaction among the factors.
order = len(factors)
terms = [term for size in range(1, order + 1) for term in itertools.combinations(factors, size)]
# C(…, Sum) is sum-to-zero coding: each effect is measured against the average of the
# levels, which is what Type III sums of squares need. Q("…") quotes a column name.
coded = {term: ":".join(f'C(Q("{name}"), Sum)' for name in term) for term in terms}
fit = smf.ols(f'Q("{response}") ~ ' + " + ".join(coded.values()), data=data).fit()
# Type III needs data in every combination of levels the model contains. With an empty
# cell some effects cannot be estimated, and statsmodels would print numbers that mean
# nothing, so stop instead (the app then tests only what the data can estimate).
if np.linalg.matrix_rank(fit.model.exog) < fit.model.exog.shape[1]:
    raise SystemExit("A combination of levels in the model has no data: choose a smaller model.")

## Descriptive statistics

In [ ]:
# One table per term of the model: each level, or each combination of levels.
def describe(values, term):
    cells = values.groupby([data[name] for name in term], observed=True)
    table = pd.DataFrame({"n": cells.count(), "mean": cells.mean(), "SD": cells.std()})
    table["mean - SD"] = table["mean"] - table["SD"]
    table["mean + SD"] = table["mean"] + table["SD"]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    table["mean - SE"] = table["mean"] - table["SE"]
    table["mean + SE"] = table["mean"] + table["SE"]
    return table


descriptive = {" * ".join(term): describe(data[response], term) for term in terms}
for name, table in descriptive.items():
    print(f"\n{name}")
    print(table.to_string())

## Test for homogeneity of variance

In [ ]:
# Levene's test (deviations from the mean) across the cells of all the factors, on the
# model's residuals: what the model assumes is that its error has one variance.
cells = [group.to_numpy() for _, group in fit.resid.groupby([data[name] for name in factors], observed=True)]
levene = stats.levene(*cells, center="mean")
print(f"\nLevene: F = {levene.statistic:.4f}, df = ({len(cells) - 1}, {len(data) - len(cells)}), "
      f"P = {levene.pvalue:.4g}")

## Analysis of variance table

In [ ]:
# Type III sums of squares: each term is tested as if it were added last, so the order of
# the factors does not matter even when the cells have different sizes.
type3 = anova_lm(fit, typ=3).loc[list(coded.values())]
anova = pd.DataFrame(
    {"SS": type3["sum_sq"].to_numpy(), "df": type3["df"].to_numpy(),
     "MS": (type3["sum_sq"] / type3["df"]).to_numpy(), "F": type3["F"].to_numpy(),
     "P": type3["PR(>F)"].to_numpy()},
    index=list(descriptive),
)
anova.loc["Error"] = [fit.ssr, fit.df_resid, fit.ssr / fit.df_resid, np.nan, np.nan]
anova.loc["Total"] = [fit.centered_tss, len(data) - 1, np.nan, np.nan, np.nan]
print()
print(anova)

## Effect sizes

In [ ]:
# For each term of the table: eta² = SS / SS total; partial eta² = SS / (SS + SS error),
# the term against the error alone; omega² = (SS - df·MS error) / (SS total + MS error),
# which removes eta²'s upward bias (Hays 1963); and partial omega² = (SS - df·MS error)
# / (SS + (N - df)·MS error) (Olejnik & Algina 2003). Partial eta² carries the 95%
# interval that inverts the noncentral F at the term's F (Steiger 2004), written out in
# eta_squared_interval.py.
# partial_eta_squared_interval: defined in the helper cell eta_squared_interval above (eta_squared_interval.py in the code download)


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


ss_error, df_error = fit.ssr, fit.df_resid
ms_error = ss_error / df_error
ss_total, n_total = fit.centered_tss, len(data)
rows = []
for name in anova.index[:-2]:
    ss, df, f_value = anova.loc[name, ["SS", "df", "F"]]
    lower, upper = partial_eta_squared_interval(f_value, df, df_error, 0.95)
    rows += [[name, "eta²", ss / ss_total, np.nan, np.nan],
             [name, "partial eta²", ss / (ss + ss_error), lower, upper],
             [name, "omega²", (ss - df * ms_error) / (ss_total + ms_error), np.nan, np.nan],
             [name, "partial omega²", (ss - df * ms_error) / (ss + (n_total - df) * ms_error),
              np.nan, np.nan]]
effect_sizes = pd.DataFrame(rows, columns=["effect", "measure", "value", "lower", "upper"])
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes.to_string(index=False))

## Normality of the residuals: the Shapiro-Wilk test

In [ ]:
# The model's residuals -- the errors it assumes normal -- rather than the responses,
# which mix the cells' different means.
shapiro = stats.shapiro(fit.resid)
normality = [len(fit.resid), shapiro.statistic, shapiro.pvalue]
print(f"\nShapiro-Wilk on the residuals: n = {len(fit.resid)}, W = {shapiro.statistic:.4f}, "
      f"P = {shapiro.pvalue:.4g}")

## What the result says

Factorial analysis of variance tests several factors at once, and their interactions (whether one factor's effect depends on another). F compares an effect with the noise, and the P value is the chance of an effect at least this large if the factor did nothing.

Price tag has F = 24 and a P value far below 0.01: the same pizza is rated 7.5 on average with a \$14 card and 6.17 with an \$8 card. Cheese has F = 13.5 and P = 0.001: extra cheese scores 7.33 against 6.33. Crust has F = 1.5 and P = 0.243, so no crust difference is shown. No interaction is significant.

The price on the card and extra cheese both raise the rating. The crust can be chosen on cost.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.